# TF-IDF, SVD and MLP model selection

This notebook compares direct TF-IDF MLPs, weighted-SVD representations with classical classifiers, and weighted-SVD MLPs. Model selection uses the training and validation sets only; the test set is not used here.

In [1]:
# 1) Imports, paths and experiment settings
from __future__ import annotations

import copy, inspect, itertools, json, math, os, random, shutil, sys, time, warnings
from pathlib import Path
from typing import Any, Dict, List, Mapping, Sequence, Tuple

# Strict deterministic CUDA/cuBLAS and known-good one-thread native numerical behavior.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import joblib
from joblib import Parallel, delayed, parallel_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.svm import LinearSVC, SVC
import sklearn

import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.experiment_sampling import BalancedEpochSampler, stable_u64
from scripts.modeling_protocol_final import (
    TOPICS, PORTALS, TEXT_FIELD,
    TFIDF_FIXED, TFIDF_GRID, SVD_COMPONENTS, SVD_SEED, SVD_N_ITER,
    CLASSICAL_MODEL_GRID, CLASSICAL_FAMILIES, CLASSICAL_BACKEND, CLASSICAL_N_JOBS, CLASSICAL_INNER_THREADS,
    MLP_SEARCH_MASTER_SEED, MLP_CONFIRMATION_MASTER_SEEDS, MLP_CANONICAL_STAGE1,
    MLP_ARCHITECTURES, MLP_DROPOUTS, MLP_LEARNING_RATES, MLP_WEIGHT_DECAYS,
    MLP_BATCH_SIZE, MLP_MAX_EPOCHS, MLP_MIN_EPOCHS, MLP_PATIENCE, MLP_MIN_DELTA,
    DIRECT_STAGE1_KEEP, SVD_STAGE1_KEEP, MLP_STAGE2_KEEP,
)

_sampler_sig = inspect.signature(BalancedEpochSampler.__init__)
for _kw in ("sampler_seed", "global_shuffle_seed"):
    if _kw not in _sampler_sig.parameters:
        raise RuntimeError("Update scripts/experiment_sampling.py to the backward-compatible seeded sampler before running.")

CORPUS_PATH = PROJECT_ROOT / "frozen_data" / "corpus_frozen.jsonl"
MANIFEST_PATH = PROJECT_ROOT / "experiment_data" / "experiment_manifest.jsonl"
TRAIN_BUCKETS_PATH = PROJECT_ROOT / "experiment_data" / "train_buckets.json"
OUTPUT_DIR = PROJECT_ROOT / "modeling_results" / "tfidf_neural_svd_final"

LABEL_TO_ID = {p:i for i,p in enumerate(PORTALS)}
ID_TO_LABEL = {i:p for p,i in LABEL_TO_ID.items()}
SEARCH_MASTER_SEED = MLP_SEARCH_MASTER_SEED
CONFIRMATION_MASTER_SEEDS = MLP_CONFIRMATION_MASTER_SEEDS
CANONICAL_STAGE1 = MLP_CANONICAL_STAGE1
ARCHITECTURES = MLP_ARCHITECTURES
DROPOUTS = MLP_DROPOUTS
LEARNING_RATES = MLP_LEARNING_RATES
WEIGHT_DECAYS = MLP_WEIGHT_DECAYS
BATCH_SIZE = MLP_BATCH_SIZE
MAX_EPOCHS = MLP_MAX_EPOCHS
MIN_EPOCHS = MLP_MIN_EPOCHS
EARLY_STOPPING_PATIENCE = MLP_PATIENCE
EARLY_STOPPING_MIN_DELTA = MLP_MIN_DELTA
STAGE2_KEEP_CONFIGS = MLP_STAGE2_KEEP
MLP_EVAL_BATCH_SIZE = 256
RUN_TORCH_LINEAR_ABLATION = True
VERBOSE_EPOCHS = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("TF-IDF configs:", len(TFIDF_GRID), "SVD ranks:", SVD_COMPONENTS, "classical configs/rep:", len(CLASSICAL_MODEL_GRID))


Project root: /home/dorjan/NLP/nlp_portal_project
Device: cuda
TF-IDF configs: 16 SVD ranks: [64, 128, 256, 512] classical configs/rep: 91


In [2]:
# 2) Shared utilities and reproducibility settings

def seed32(seed: int, *parts: object) -> int:
    return stable_u64(seed, *parts) % (2**31 - 1)


def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)


def atomic_csv_dump(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def json_load(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def save_figure(fig: plt.Figure, path: Path, dpi: int = 170) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")


def set_torch_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

for _p in [CORPUS_PATH, MANIFEST_PATH, TRAIN_BUCKETS_PATH]:
    if not _p.exists():
        raise FileNotFoundError(_p)

# Kept only as an internal compatibility identifier so already-completed caches/run records resume normally.
PROTOCOL_HASH = '936e424d488c95ab4c3b8df23af51aea4759ceb3e380139958cf1876d70057c9'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# 3) Load frozen manifest and ONLY train/validation article text
manifest_rows=[]
with MANIFEST_PATH.open("r",encoding="utf-8") as f:
    for line in f:
        r=json.loads(line)
        if r.get("experiment_eligible") and r.get("split") in {"train","validation","test"}: manifest_rows.append(r)
manifest_all=pd.DataFrame(manifest_rows)
expected_counts={("train","topicA"):811,("validation","topicA"):272,("test","topicA"):281,("train","topicB"):721,("validation","topicB"):245,("test","topicB"):237}
assert manifest_all.groupby(["split","topic"]).size().to_dict()==expected_counts
assert manifest_all.article_id.is_unique
model_manifest=manifest_all[manifest_all.split.isin(["train","validation"])].copy()
allowed_ids=set(model_manifest.article_id)
text_by_id={}
with CORPUS_PATH.open("r",encoding="utf-8") as f:
    for line in f:
        r=json.loads(line); aid=r.get("article_id")
        if aid in allowed_ids: text_by_id[aid]=r[TEXT_FIELD]
assert set(text_by_id)==allowed_ids
model_manifest["text"]=model_manifest.article_id.map(text_by_id)

TOPIC_DATA={}; summary=[]
for topic in TOPICS:
    tr=model_manifest[(model_manifest.topic==topic)&(model_manifest.split=="train")].sort_values("article_id").reset_index(drop=True)
    va=model_manifest[(model_manifest.topic==topic)&(model_manifest.split=="validation")].sort_values("article_id").reset_index(drop=True)
    tr["label_id"]=tr.portal.map(LABEL_TO_ID).astype(int); va["label_id"]=va.portal.map(LABEL_TO_ID).astype(int)
    TOPIC_DATA[topic]={"train":tr,"validation":va,"train_index_by_id":{a:i for i,a in enumerate(tr.article_id)}}
    for split,df in [("train",tr),("validation",va)]:
        for portal in PORTALS:
            s=df[df.portal==portal]; summary.append({"topic":topic,"split":split,"portal":portal,"raw_n":len(s),"effective_weight_mass":float(s.balance_weight.sum())})
DATA_SPLIT_SUMMARY=pd.DataFrame(summary); atomic_csv_dump(DATA_SPLIT_SUMMARY,OUTPUT_DIR/"DATA_SPLIT_SUMMARY.csv"); display(DATA_SPLIT_SUMMARY)
print("TEST TEXT LOADED:",False)


,topic,split,portal,raw_n,effective_weight_mass
0,topicA,train,aljazeera,332,181.0
1,topicA,train,bbc,207,181.0
2,topicA,train,guardian,272,181.0
3,topicA,validation,aljazeera,111,62.0
4,topicA,validation,bbc,68,62.0
5,topicA,validation,guardian,93,62.0
6,topicB,train,aljazeera,112,109.0
7,topicB,train,bbc,194,109.0
8,topicB,train,guardian,415,109.0
9,topicB,validation,aljazeera,38,37.0


TEST TEXT LOADED: False


In [4]:
# 4) Metrics and balanced-sampler helpers

def weighted_metrics_from_predictions(y_true, y_pred, sample_weight):
    labels = np.arange(len(PORTALS))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, sample_weight=sample_weight, zero_division=0)
    return {
        "weighted_macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", sample_weight=sample_weight, zero_division=0)),
        "weighted_accuracy": float(accuracy_score(y_true, y_pred, sample_weight=sample_weight)),
        "raw_macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "raw_accuracy": float(accuracy_score(y_true, y_pred)),
        "weighted_f1_by_class": {PORTALS[i]: float(f[i]) for i in labels},
        "weighted_confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels, sample_weight=sample_weight).tolist(),
        "raw_confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def cross_entropy_from_logits(logits, y_true, weights=None):
    z = logits.astype(np.float64, copy=False)
    zm = z.max(axis=1, keepdims=True)
    lse = zm[:, 0] + np.log(np.exp(z - zm).sum(axis=1))
    losses = lse - z[np.arange(len(y_true)), y_true]
    return float(losses.mean() if weights is None else np.average(losses, weights=weights))


def evaluate_logits(logits, y, w):
    out = weighted_metrics_from_predictions(y, logits.argmax(1), w)
    out["weighted_cross_entropy"] = cross_entropy_from_logits(logits, y, w)
    out["raw_cross_entropy"] = cross_entropy_from_logits(logits, y, None)
    return out


def checkpoint_is_better(cand, inc):
    if inc is None:
        return True
    eps = 1e-12
    if cand["weighted_macro_f1"] > inc["weighted_macro_f1"] + eps:
        return True
    if abs(cand["weighted_macro_f1"] - inc["weighted_macro_f1"]) <= eps:
        if cand["weighted_accuracy"] > inc["weighted_accuracy"] + eps:
            return True
        if abs(cand["weighted_accuracy"] - inc["weighted_accuracy"]) <= eps and cand["weighted_cross_entropy"] < inc["weighted_cross_entropy"] - eps:
            return True
    return False


def derived_training_seeds(master_seed, topic):
    return {
        "model_seed": seed32(master_seed, "model-init", topic),
        "sampler_seed": stable_u64(master_seed, "fair-cycle", topic),
        "shuffle_seed": stable_u64(master_seed, "global-shuffle", topic),
    }


## TF-IDF and SVD representations

Each TF-IDF vectorizer is fitted once per topic on the source-topic training articles, without sample weighting. The same fitted matrices are reused by the direct MLP and SVD branches. SVD is fitted with the frozen training balance weights.

In [5]:
# 5) TF-IDF and weighted-SVD caches

def tfidf_config_id(cfg):
    ng=tuple(cfg["ngram_range"]); return f"ng{ng[0]}{ng[1]}_df{int(cfg['min_df'])}_sub{int(bool(cfg['sublinear_tf']))}"

def make_vectorizer(cfg):
    return TfidfVectorizer(**TFIDF_FIXED,ngram_range=tuple(cfg["ngram_range"]),min_df=int(cfg["min_df"]),sublinear_tf=bool(cfg["sublinear_tf"]),dtype=np.float64)

def direct_rep_id(cfg): return "direct__"+tfidf_config_id(cfg)
def svd_rep_id(cfg,k): return f"svd__{tfidf_config_id(cfg)}__k{int(k):04d}"

def tfidf_cache_dir(topic,tid): return OUTPUT_DIR/"representation_cache"/"tfidf"/topic/tid

def build_tfidf_rep(topic,cfg,force=False):
    tid=tfidf_config_id(cfg); d=tfidf_cache_dir(topic,tid); trp=d/"train.npz"; vap=d/"validation.npz"; vp=d/"vectorizer.joblib"; mp=d/"metadata.json"
    tr=TOPIC_DATA[topic]["train"]; va=TOPIC_DATA[topic]["validation"]
    if all(p.exists() for p in [trp,vap,vp,mp]) and not force:
        meta=json_load(mp)
        if meta.get("protocol_hash")==PROTOCOL_HASH:
            return {"rep_id":direct_rep_id(cfg),"family":"direct","X_train":sparse.load_npz(trp),"X_val":sparse.load_npz(vap),"meta":meta}
    t0=time.time(); vec=make_vectorizer(cfg); Xtr=vec.fit_transform(tr.text.tolist()).tocsr(); Xva=vec.transform(va.text.tolist()).tocsr()
    d.mkdir(parents=True,exist_ok=True); sparse.save_npz(trp,Xtr,compressed=True); sparse.save_npz(vap,Xva,compressed=True); joblib.dump(vec,vp,compress=3)
    meta={"protocol_hash":PROTOCOL_HASH,"topic":topic,"rep_id":direct_rep_id(cfg),"tfidf_config_id":tid,"tfidf_config":{**cfg,"ngram_range":list(cfg["ngram_range"])},"n_tfidf_features":int(Xtr.shape[1]),"build_seconds":time.time()-t0}
    atomic_json_dump(meta,mp); return {"rep_id":direct_rep_id(cfg),"family":"direct","X_train":Xtr,"X_val":Xva,"meta":meta}

def load_tfidf_rep(topic,rep_id):
    tid=rep_id.replace("direct__","",1); d=tfidf_cache_dir(topic,tid); meta=json_load(d/"metadata.json"); assert meta["protocol_hash"]==PROTOCOL_HASH
    return {"rep_id":rep_id,"family":"direct","X_train":sparse.load_npz(d/"train.npz"),"X_val":sparse.load_npz(d/"validation.npz"),"meta":meta}

def cfg_from_tfidf_id(tid):
    for cfg in TFIDF_GRID:
        if tfidf_config_id(cfg)==tid:return dict(cfg)
    raise KeyError(tid)

def svd_cache_dir(topic,rep_id): return OUTPUT_DIR/"representation_cache"/"svd"/topic/rep_id

def build_svd_rep(topic,cfg,k,force=False):
    rid=svd_rep_id(cfg,k); d=svd_cache_dir(topic,rid); zp=d/"dense.npz"; sp=d/"svd.joblib"; mp=d/"metadata.json"
    if all(p.exists() for p in [zp,sp,mp]) and not force:
        meta=json_load(mp)
        if meta.get("protocol_hash")==PROTOCOL_HASH:
            z=np.load(zp); return {"rep_id":rid,"family":"svd","X_train":z["X_train"],"X_val":z["X_val"],"meta":meta}
    direct=build_tfidf_rep(topic,cfg); Xtr=direct["X_train"]; Xva=direct["X_val"]
    if int(k)>=min(Xtr.shape): raise ValueError((topic,rid,Xtr.shape))
    t0=time.time(); w=TOPIC_DATA[topic]["train"].balance_weight.to_numpy(np.float64); Xw=Xtr.multiply(np.sqrt(w)[:,None])
    svd=TruncatedSVD(n_components=int(k),algorithm="randomized",n_iter=SVD_N_ITER,random_state=SVD_SEED); svd.fit(Xw)
    Ztr=svd.transform(Xtr).astype(np.float32,copy=False); Zva=svd.transform(Xva).astype(np.float32,copy=False)
    d.mkdir(parents=True,exist_ok=True); np.savez_compressed(zp,X_train=Ztr,X_val=Zva); joblib.dump(svd,sp,compress=3)
    meta={"protocol_hash":PROTOCOL_HASH,"topic":topic,"rep_id":rid,"tfidf_config_id":tfidf_config_id(cfg),"tfidf_config":{**cfg,"ngram_range":list(cfg["ngram_range"])},"svd_components":int(k),"svd_seed":SVD_SEED,"svd_n_iter":SVD_N_ITER,"weighted_svd":True,"n_tfidf_features":int(Xtr.shape[1]),"explained_variance_ratio_sum":float(svd.explained_variance_ratio_.sum()),"top_singular_values":[float(x) for x in svd.singular_values_[:20]],"train_shape":list(Ztr.shape),"validation_shape":list(Zva.shape),"build_seconds":time.time()-t0}
    atomic_json_dump(meta,mp); return {"rep_id":rid,"family":"svd","X_train":Ztr,"X_val":Zva,"meta":meta}

def load_svd_rep(topic,rep_id):
    d=svd_cache_dir(topic,rep_id); meta=json_load(d/"metadata.json"); assert meta["protocol_hash"]==PROTOCOL_HASH; z=np.load(d/"dense.npz")
    return {"rep_id":rep_id,"family":"svd","X_train":z["X_train"],"X_val":z["X_val"],"meta":meta}

def load_representation(topic,rep_id):
    return load_tfidf_rep(topic,rep_id) if rep_id.startswith("direct__") else load_svd_rep(topic,rep_id)

TFIDF_REP_IDS={topic:[] for topic in TOPICS}; tfidf_meta=[]
for topic in TOPICS:
    for cfg in TFIDF_GRID:
        rep=build_tfidf_rep(topic,cfg); TFIDF_REP_IDS[topic].append(rep["rep_id"]); tfidf_meta.append(rep["meta"])
_tfidf_summary_path = OUTPUT_DIR/"TFIDF_REPRESENTATION_SUMMARY.csv"
if _tfidf_summary_path.exists():
    TFIDF_METADATA = pd.read_csv(_tfidf_summary_path)
else:
    TFIDF_METADATA = pd.DataFrame(tfidf_meta)
    atomic_csv_dump(TFIDF_METADATA, _tfidf_summary_path)
display(TFIDF_METADATA[["topic","tfidf_config_id","n_tfidf_features","build_seconds"]])


,topic,tfidf_config_id,n_tfidf_features,build_seconds
0,topicA,ng11_df2_sub0,11108,0.527251
1,topicA,ng11_df2_sub1,11108,0.433338
2,topicA,ng11_df3_sub0,8660,0.409858
3,topicA,ng11_df3_sub1,8660,0.418666
4,topicA,ng11_df5_sub0,6268,0.388752
5,topicA,ng11_df5_sub1,6268,0.401252
6,topicA,ng11_df10_sub0,4066,0.380605
7,topicA,ng11_df10_sub1,4066,0.387132
8,topicA,ng12_df2_sub0,70356,1.486670
9,topicA,ng12_df2_sub1,70356,1.505854


In [6]:
# 6) Strict-deterministic MLP training that supports sparse direct TF-IDF or dense SVD inputs
class MLPClassifier(nn.Module):
    def __init__(self,input_dim,hidden_layers,dropout,n_classes=3):
        super().__init__(); layers=[]; prev=int(input_dim)
        for h in hidden_layers:
            layers += [nn.Linear(prev,int(h)),nn.ReLU()]
            if dropout>0: layers.append(nn.Dropout(float(dropout)))
            prev=int(h)
        layers.append(nn.Linear(prev,n_classes)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def count_trainable_parameters(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

def feature_dim(X): return int(X.shape[1])
def matrix_rows_to_tensor(X,idx):
    part=X[idx]
    arr=part.toarray() if sparse.issparse(part) else np.asarray(part)
    return torch.from_numpy(np.asarray(arr,dtype=np.float32)).to(DEVICE)

def model_logits_matrix(model,X,batch_size=MLP_EVAL_BATCH_SIZE):
    model.eval(); outs=[]
    with torch.no_grad():
        for start in range(0,X.shape[0],batch_size): outs.append(model(matrix_rows_to_tensor(X,slice(start,min(start+batch_size,X.shape[0])))).cpu().numpy())
    return np.concatenate(outs,axis=0)

def train_one_run(topic,rep,hidden_layers,dropout,learning_rate,weight_decay,master_seed,run_id,history_path,checkpoint_path=None,predictions_path=None,verbose_epochs=VERBOSE_EPOCHS):
    tr=TOPIC_DATA[topic]["train"]; va=TOPIC_DATA[topic]["validation"]; idx_by_id=TOPIC_DATA[topic]["train_index_by_id"]
    seeds=derived_training_seeds(master_seed,topic); set_torch_reproducibility(seeds["model_seed"])
    sampler=BalancedEpochSampler(TRAIN_BUCKETS_PATH,sampler_seed=seeds["sampler_seed"],global_shuffle_seed=seeds["shuffle_seed"]); epoch_size=sampler.expected_epoch_size(topic)
    Xtr=rep["X_train"]; Xva=rep["X_val"]; ytr=tr.label_id.to_numpy(np.int64); yva=va.label_id.to_numpy(np.int64); trw=tr.balance_weight.to_numpy(np.float64); vaw=va.balance_weight.to_numpy(np.float64)
    model=MLPClassifier(feature_dim(Xtr),hidden_layers,dropout,len(PORTALS)).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=float(learning_rate),weight_decay=float(weight_decay),betas=(0.9,0.999))
    history=[]; best_metrics=None; best_state=None; best_epoch=None; anchor=-np.inf; since=0; t0=time.time()
    for epoch in range(MAX_EPOCHS):
        model.train(); ordered=sampler.article_ids_for_epoch(epoch,topic); idx=np.fromiter((idx_by_id[a] for a in ordered),dtype=np.int64,count=len(ordered)); loss_sum=0.0; n=0
        for start in range(0,len(idx),BATCH_SIZE):
            bi=idx[start:start+BATCH_SIZE]; xb=matrix_rows_to_tensor(Xtr,bi); yb=torch.from_numpy(ytr[bi]).to(DEVICE); opt.zero_grad(set_to_none=True); logits=model(xb); loss=F.cross_entropy(logits,yb)
            if not torch.isfinite(loss): raise FloatingPointError((run_id,epoch+1))
            loss.backward(); opt.step(); loss_sum += float(loss.detach().cpu())*len(bi); n+=len(bi)
        tr_logits=model_logits_matrix(model,Xtr); va_logits=model_logits_matrix(model,Xva); tre=evaluate_logits(tr_logits,ytr,trw); vae=evaluate_logits(va_logits,yva,vaw)
        row={"epoch":epoch+1,"sampled_train_loss":loss_sum/n,"full_train_weighted_macro_f1":tre["weighted_macro_f1"],"full_train_weighted_accuracy":tre["weighted_accuracy"],"full_train_weighted_cross_entropy":tre["weighted_cross_entropy"],"val_weighted_macro_f1":vae["weighted_macro_f1"],"val_weighted_accuracy":vae["weighted_accuracy"],"val_weighted_cross_entropy":vae["weighted_cross_entropy"],"val_raw_macro_f1":vae["raw_macro_f1"],"val_raw_accuracy":vae["raw_accuracy"]}; history.append(row)
        if checkpoint_is_better(vae,best_metrics): best_metrics=copy.deepcopy(vae); best_epoch=epoch+1; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        cur=vae["weighted_macro_f1"]
        if cur>=anchor+EARLY_STOPPING_MIN_DELTA: anchor=cur; since=0
        else: since+=1
        if verbose_epochs: print(run_id,epoch+1,round(cur,4))
        if epoch+1>=MIN_EPOCHS and since>=EARLY_STOPPING_PATIENCE: break
    assert best_state is not None; atomic_csv_dump(pd.DataFrame(history),history_path); model.load_state_dict(best_state); logits=model_logits_matrix(model,Xva); check=evaluate_logits(logits,yva,vaw); assert abs(check["weighted_macro_f1"]-best_metrics["weighted_macro_f1"])<1e-10
    if checkpoint_path is not None:
        checkpoint_path.parent.mkdir(parents=True,exist_ok=True); torch.save({"state_dict":best_state,"topic":topic,"rep_id":rep["rep_id"],"representation_family":rep["family"],"input_dim":feature_dim(Xtr),"hidden_layers":list(hidden_layers),"dropout":float(dropout),"learning_rate":float(learning_rate),"weight_decay":float(weight_decay),"master_seed":int(master_seed),"derived_seeds":seeds,"best_epoch":int(best_epoch),"best_validation_metrics":best_metrics,"protocol_hash":PROTOCOL_HASH},checkpoint_path)
    if predictions_path is not None:
        pred=logits.argmax(1); pdf=va[["article_id","portal","year_month","length_bin","balance_weight"]].copy(); pdf["predicted_portal"]=[ID_TO_LABEL[int(x)] for x in pred]
        for i,p in enumerate(PORTALS): pdf[f"logit_{p}"]=logits[:,i]
        atomic_csv_dump(pdf,predictions_path)
    return {"run_id":run_id,"topic":topic,"rep_id":rep["rep_id"],"representation_family":rep["family"],"hidden_layers":list(hidden_layers),"dropout":float(dropout),"learning_rate":float(learning_rate),"weight_decay":float(weight_decay),"master_seed":int(master_seed),"model_seed":int(seeds["model_seed"]),"sampler_seed":int(seeds["sampler_seed"]),"shuffle_seed":int(seeds["shuffle_seed"]),"n_parameters":count_trainable_parameters(model),"input_dim":feature_dim(Xtr),"epoch_size":epoch_size,"best_epoch":int(best_epoch),"epochs_ran":len(history),"best_val_weighted_macro_f1":float(best_metrics["weighted_macro_f1"]),"best_val_weighted_accuracy":float(best_metrics["weighted_accuracy"]),"best_val_weighted_cross_entropy":float(best_metrics["weighted_cross_entropy"]),"best_val_raw_macro_f1":float(best_metrics["raw_macro_f1"]),"best_val_weighted_f1_by_class":best_metrics["weighted_f1_by_class"],"best_val_weighted_confusion_matrix":best_metrics["weighted_confusion_matrix"],"best_val_raw_confusion_matrix":best_metrics["raw_confusion_matrix"],"seconds":time.time()-t0,"history_path":str(history_path.relative_to(OUTPUT_DIR)),"checkpoint_path":str(checkpoint_path.relative_to(OUTPUT_DIR)) if checkpoint_path else None,"predictions_path":str(predictions_path.relative_to(OUTPUT_DIR)) if predictions_path else None}
print("MLP utilities ready.")


MLP utilities ready.


In [7]:
# 7) Generic three-stage MLP selection helpers

def run_record_path(stage,run_id): return OUTPUT_DIR/stage/"run_records"/f"{run_id}.json"
def history_path(stage,run_id): return OUTPUT_DIR/stage/"histories"/f"{run_id}.csv"
def load_stage_records(stage):
    folder=OUTPUT_DIR/stage/"run_records"; recs=[json_load(p) for p in sorted(folder.glob("*.json"))] if folder.exists() else []; return pd.DataFrame(recs)
def config_slug(hidden,dropout,lr,wd):
    arch="x".join(map(str,hidden)) if hidden else "linear"; compact=lambda x:f"{x:.0e}".replace("+","").replace("-","m"); return f"h{arch}_d{str(dropout).replace('.','p')}_lr{compact(lr)}_wd{compact(wd)}"
def rank_run_records(df): return df.sort_values(["best_val_weighted_macro_f1","best_val_weighted_accuracy","best_val_weighted_cross_entropy","grid_order"],ascending=[False,False,True,True],kind="mergesort").reset_index(drop=True)

def plot_confusion(arr,title,path):
    arr=np.asarray(arr,float); fig,ax=plt.subplots(figsize=(5.5,4.7)); im=ax.imshow(arr); ax.set_xticks(range(3),PORTALS,rotation=30,ha="right"); ax.set_yticks(range(3),PORTALS); ax.set_xlabel("Predicted portal"); ax.set_ylabel("Actual portal"); ax.set_title(title)
    for i in range(3):
        for j in range(3): ax.text(j,i,f"{arr[i,j]:.1f}",ha="center",va="center")
    fig.colorbar(im,ax=ax,fraction=.046,pad=.04); save_figure(fig,path); plt.close(fig)

def run_mlp_stage1(family,rep_ids_by_topic,keep_n):
    stage=f"{family}_mlp_stage1"; total_expected=0
    for topic in TOPICS:
        for order,rid in enumerate(rep_ids_by_topic[topic]):
            total_expected+=1; run_id=f"{topic}__{rid}__canonical"; rp=run_record_path(stage,run_id)
            if rp.exists(): continue
            print(f"[{family} Stage1][{topic}] {order+1}/{len(rep_ids_by_topic[topic])}: {rid}"); rep=load_representation(topic,rid)
            rec=train_one_run(topic,rep,CANONICAL_STAGE1["hidden_layers"],CANONICAL_STAGE1["dropout"],CANONICAL_STAGE1["learning_rate"],CANONICAL_STAGE1["weight_decay"],SEARCH_MASTER_SEED,run_id,history_path(stage,run_id))
            rec.update({"stage":stage,"grid_order":order,"tfidf_config_id":rep["meta"]["tfidf_config_id"],"n_tfidf_features":rep["meta"]["n_tfidf_features"],"svd_components":rep["meta"].get("svd_components"),"svd_explained_variance_ratio_sum":rep["meta"].get("explained_variance_ratio_sum")}); atomic_json_dump(rec,rp)
            print(f"  val_wF1={rec['best_val_weighted_macro_f1']:.4f} epoch={rec['best_epoch']} ({rec['seconds']:.1f}s)")
    df=load_stage_records(stage); atomic_csv_dump(df.sort_values(["topic","grid_order"]),OUTPUT_DIR/stage/"all_results.csv")
    short={};
    for topic in TOPICS:
        s=rank_run_records(df[df.topic==topic]); top=s.head(keep_n); short[topic]=top.rep_id.tolist(); atomic_csv_dump(top,OUTPUT_DIR/stage/f"{topic}_top{keep_n}.csv")
        fig,ax=plt.subplots(figsize=(10,6)); q=s.head(min(20,len(s))).iloc[::-1]; ax.barh(q.rep_id,q.best_val_weighted_macro_f1); ax.set_xlabel("Validation weighted macro-F1"); ax.set_title(f"{topic}: {family} Stage 1 top representations"); save_figure(fig,OUTPUT_DIR/stage/"plots"/f"{topic}_top.png"); plt.close(fig)
    atomic_json_dump(short,OUTPUT_DIR/stage/"shortlist.json"); return df,short

def run_mlp_stage2(family,short):
    stage=f"{family}_mlp_stage2"; per=len(ARCHITECTURES)*len(DROPOUTS)*len(LEARNING_RATES)*len(WEIGHT_DECAYS)
    for topic in TOPICS:
        order=0
        for rid in short[topic]:
            rep=load_representation(topic,rid)
            for hidden,dropout,lr,wd in itertools.product(ARCHITECTURES,DROPOUTS,LEARNING_RATES,WEIGHT_DECAYS):
                slug=config_slug(hidden,dropout,lr,wd); run_id=f"{topic}__{rid}__{slug}"; rp=run_record_path(stage,run_id)
                if not rp.exists():
                    print(f"[{family} Stage2][{topic}] {order+1}/{len(short[topic])*per}: {rid} | {slug}")
                    rec=train_one_run(topic,rep,hidden,dropout,lr,wd,SEARCH_MASTER_SEED,run_id,history_path(stage,run_id)); rec.update({"stage":stage,"grid_order":order}); atomic_json_dump(rec,rp)
                order+=1
    df=load_stage_records(stage); atomic_csv_dump(df.sort_values(["topic","grid_order"]),OUTPUT_DIR/stage/"all_results.csv"); configs={}
    for topic in TOPICS:
        s=rank_run_records(df[df.topic==topic]); top=s.head(STAGE2_KEEP_CONFIGS); out=[]
        for _,r in top.iterrows(): out.append({"stage2_run_id":r.run_id,"rep_id":r.rep_id,"hidden_layers":list(r.hidden_layers),"dropout":float(r.dropout),"learning_rate":float(r.learning_rate),"weight_decay":float(r.weight_decay)})
        configs[topic]=out; atomic_csv_dump(top,OUTPUT_DIR/stage/f"{topic}_top{STAGE2_KEEP_CONFIGS}.csv")
    atomic_json_dump(configs,OUTPUT_DIR/stage/"shortlist.json"); return df,configs

def run_mlp_stage3(family,configs):
    stage=f"{family}_mlp_stage3"
    for topic in TOPICS:
        order=0
        for ci,cfg in enumerate(configs[topic]):
            rep=load_representation(topic,cfg["rep_id"])
            for si,seed in enumerate(CONFIRMATION_MASTER_SEEDS):
                slug=config_slug(cfg["hidden_layers"],cfg["dropout"],cfg["learning_rate"],cfg["weight_decay"]); config_id=f"cfg{ci+1:02d}__{cfg['rep_id']}__{slug}"; run_id=f"{topic}__{config_id}__seed{seed}"; rp=run_record_path(stage,run_id)
                if not rp.exists():
                    print(f"[{family} Stage3][{topic}] {order+1}/{len(configs[topic])*len(CONFIRMATION_MASTER_SEEDS)}: {config_id} seed={seed}")
                    cp=OUTPUT_DIR/stage/"checkpoints"/f"{run_id}.pt"; pp=OUTPUT_DIR/stage/"validation_predictions"/f"{run_id}.csv"
                    rec=train_one_run(topic,rep,cfg["hidden_layers"],cfg["dropout"],cfg["learning_rate"],cfg["weight_decay"],seed,run_id,history_path(stage,run_id),cp,pp)
                    rec.update({"stage":stage,"grid_order":order,"config_index":ci,"config_id":config_id,"seed_index":si,"stage2_run_id":cfg["stage2_run_id"]}); atomic_json_dump(rec,rp)
                order+=1
    df=load_stage_records(stage); atomic_csv_dump(df.sort_values(["topic","config_index","seed_index"]),OUTPUT_DIR/stage/"all_seed_runs.csv"); return df

def aggregate_mlp_stage3(family,df):
    rows=[]; selected={}; stage=f"{family}_mlp_stage3"
    for topic in TOPICS:
        for cid,g in df[df.topic==topic].groupby("config_id",sort=False):
            assert set(g.master_seed.astype(int))==set(CONFIRMATION_MASTER_SEEDS); r=g.iloc[0]; rows.append({"topic":topic,"config_id":cid,"rep_id":r.rep_id,"hidden_layers":list(r.hidden_layers),"dropout":float(r.dropout),"learning_rate":float(r.learning_rate),"weight_decay":float(r.weight_decay),"n_parameters":int(r.n_parameters),"mean_val_weighted_macro_f1":float(g.best_val_weighted_macro_f1.mean()),"std_val_weighted_macro_f1":float(g.best_val_weighted_macro_f1.std(ddof=1)),"mean_val_weighted_accuracy":float(g.best_val_weighted_accuracy.mean()),"mean_val_weighted_cross_entropy":float(g.best_val_weighted_cross_entropy.mean()),"mean_best_epoch":float(g.best_epoch.mean()),"config_index":int(r.config_index)})
    summary=pd.DataFrame(rows).sort_values(["topic","mean_val_weighted_macro_f1","mean_val_weighted_accuracy","mean_val_weighted_cross_entropy","n_parameters","config_index"],ascending=[True,False,False,True,True,True],kind="mergesort").reset_index(drop=True); atomic_csv_dump(summary,OUTPUT_DIR/stage/"config_summary.csv")
    for topic in TOPICS:
        s=summary[summary.topic==topic].reset_index(drop=True); winner=s.iloc[0].to_dict(); selected[topic]=winner; runs=df[(df.topic==topic)&(df.config_id==winner["config_id"])]
        conf=np.stack([np.asarray(x,float) for x in runs.best_val_weighted_confusion_matrix]).mean(0); plot_confusion(conf,f"{topic}: {family} selected MLP validation confusion",OUTPUT_DIR/stage/"plots"/f"{topic}_winner_confusion.png")
        fig,ax=plt.subplots(figsize=(8,5))
        for _,rr in runs.sort_values("master_seed").iterrows(): h=pd.read_csv(OUTPUT_DIR/rr.history_path); ax.plot(h.epoch,h.val_weighted_macro_f1,label=str(int(rr.master_seed)))
        ax.set_xlabel("Epoch"); ax.set_ylabel("Validation weighted macro-F1"); ax.set_title(f"{topic}: {family} selected MLP"); ax.legend(fontsize=8); save_figure(fig,OUTPUT_DIR/stage/"plots"/f"{topic}_winner_curves.png"); plt.close(fig)
    atomic_json_dump(selected,OUTPUT_DIR/stage/"selected_configs.json"); return summary,selected


## A. Direct TF-IDF → MLP

All TF-IDF representations are screened with the same initial MLP setting. The strongest representations enter the full MLP grid, followed by five-seed confirmation of the final candidates.

In [8]:
# 8) Direct TF-IDF → MLP final selection
DIRECT_STAGE1_DF,DIRECT_STAGE1_SHORT=run_mlp_stage1("direct",TFIDF_REP_IDS,DIRECT_STAGE1_KEEP)
DIRECT_STAGE2_DF,DIRECT_STAGE2_SHORT=run_mlp_stage2("direct",DIRECT_STAGE1_SHORT)
DIRECT_STAGE3_DF=run_mlp_stage3("direct",DIRECT_STAGE2_SHORT)
DIRECT_STAGE3_SUMMARY,DIRECT_SELECTED=aggregate_mlp_stage3("direct",DIRECT_STAGE3_DF)
display(DIRECT_STAGE3_SUMMARY)
print(json.dumps(DIRECT_SELECTED,indent=2))


,topic,config_id,rep_id,hidden_layers,dropout,learning_rate,weight_decay,n_parameters,mean_val_weighted_macro_f1,std_val_weighted_macro_f1,mean_val_weighted_accuracy,mean_val_weighted_cross_entropy,mean_best_epoch,config_index
0,topicA,cfg02__direct__ng12_df10_sub1__h128x64_d0p3_lr...,direct__ng12_df10_sub1,"[128, 64]",0.3,0.0010,0.001,1493251,0.902144,0.004218,0.901613,0.286656,14.6,1
1,topicA,cfg01__direct__ng12_df10_sub1__h128x64_d0p3_lr...,direct__ng12_df10_sub1,"[128, 64]",0.3,0.0010,0.000,1493251,0.900927,0.003849,0.900269,0.287474,11.4,0
2,topicA,cfg05__direct__ng12_df10_sub1__h128x64_d0p3_lr...,direct__ng12_df10_sub1,"[128, 64]",0.3,0.0003,0.001,1493251,0.896848,0.003476,0.896487,0.321760,25.0,4
3,topicA,cfg04__direct__ng12_df10_sub1__h128x64_d0p3_lr...,direct__ng12_df10_sub1,"[128, 64]",0.3,0.0003,0.000,1493251,0.896848,0.003476,0.896487,0.321858,25.0,3
4,topicA,cfg03__direct__ng12_df5_sub1__h128x64_d0p3_lr1...,direct__ng12_df5_sub1,"[128, 64]",0.3,0.0010,0.001,3070083,0.895467,0.009705,0.894910,0.302603,15.2,2
5,topicA,cfg06__direct__ng12_df5_sub1__h128x64_d0p3_lr1...,direct__ng12_df5_sub1,"[128, 64]",0.3,0.0010,0.000,3070083,0.894933,0.008536,0.894373,0.302734,14.8,5
6,topicB,cfg03__direct__ng12_df10_sub0__h64_d0p3_lr1em0...,direct__ng12_df10_sub0,[64],0.3,0.0010,0.000,622659,0.812422,0.004639,0.813558,0.585420,19.6,2
7,topicB,cfg04__direct__ng12_df10_sub0__h64_d0p3_lr1em0...,direct__ng12_df10_sub0,[64],0.3,0.0010,0.001,622659,0.812422,0.004639,0.813558,0.585453,19.6,3
8,topicB,cfg01__direct__ng12_df10_sub1__h64_d0p3_lr3em0...,direct__ng12_df10_sub1,[64],0.3,0.0003,0.000,622659,0.810825,0.016490,0.810942,0.773978,26.2,0
9,topicB,cfg02__direct__ng12_df10_sub1__h64_d0p3_lr3em0...,direct__ng12_df10_sub1,[64],0.3,0.0003,0.001,622659,0.810825,0.016490,0.810942,0.774005,26.2,1


{
  "topicA": {
    "topic": "topicA",
    "config_id": "cfg02__direct__ng12_df10_sub1__h128x64_d0p3_lr1em03_wd1em03",
    "rep_id": "direct__ng12_df10_sub1",
    "hidden_layers": [
      128,
      64
    ],
    "dropout": 0.3,
    "learning_rate": 0.001,
    "weight_decay": 0.001,
    "n_parameters": 1493251,
    "mean_val_weighted_macro_f1": 0.9021438154844885,
    "std_val_weighted_macro_f1": 0.004218297824425169,
    "mean_val_weighted_accuracy": 0.9016129032258065,
    "mean_val_weighted_cross_entropy": 0.28665582892126923,
    "mean_best_epoch": 14.6,
    "config_index": 1
  },
  "topicB": {
    "topic": "topicB",
    "config_id": "cfg03__direct__ng12_df10_sub0__h64_d0p3_lr1em03_wd0e00",
    "rep_id": "direct__ng12_df10_sub0",
    "hidden_layers": [
      64
    ],
    "dropout": 0.3,
    "learning_rate": 0.001,
    "weight_decay": 0.0,
    "n_parameters": 622659,
    "mean_val_weighted_macro_f1": 0.8124220804677545,
    "std_val_weighted_macro_f1": 0.00463877443715727,
    "mea

## B. Weighted SVD and classical models

Each SVD rank is evaluated with the same classical-model grid. This branch is selected independently from the MLP branches.

In [9]:
# 9) Build/reuse all 64 SVD representations per topic
SVD_REP_IDS={topic:[] for topic in TOPICS}; svd_meta=[]
for topic in TOPICS:
    for cfg in TFIDF_GRID:
        for k in SVD_COMPONENTS:
            rep=build_svd_rep(topic,cfg,k); SVD_REP_IDS[topic].append(rep["rep_id"]); svd_meta.append(rep["meta"])
_svd_summary_path = OUTPUT_DIR/"SVD_REPRESENTATION_SUMMARY.csv"
if _svd_summary_path.exists():
    SVD_METADATA = pd.read_csv(_svd_summary_path)
else:
    SVD_METADATA = pd.DataFrame(svd_meta)
    atomic_csv_dump(SVD_METADATA, _svd_summary_path)
display(SVD_METADATA[["topic","tfidf_config_id","svd_components","explained_variance_ratio_sum","build_seconds"]])


,topic,tfidf_config_id,svd_components,explained_variance_ratio_sum,build_seconds
0,topicA,ng11_df2_sub0,64,0.363483,0.384337
1,topicA,ng11_df2_sub0,128,0.496583,0.776334
2,topicA,ng11_df2_sub0,256,0.676617,1.739421
3,topicA,ng11_df2_sub0,512,0.887131,4.308930
4,topicA,ng11_df2_sub1,64,0.251583,0.382052
...,...,...,...,...,...
123,topicB,ng12_df10_sub0,512,0.926854,4.501296
124,topicB,ng12_df10_sub1,64,0.299898,0.401419
125,topicB,ng12_df10_sub1,128,0.464434,0.837517
126,topicB,ng12_df10_sub1,256,0.678437,1.948032


In [10]:
# 10) Complete classical benchmark on every SVD representation

def normalized_fit_weights(df):
    w=df.balance_weight.to_numpy(np.float64); return w/w.mean()

def make_estimator(spec):
    fam=spec["family"]; c=float(spec["C"])
    if fam=="logistic_regression": return LogisticRegression(C=c,solver="lbfgs",max_iter=5000,tol=1e-4,random_state=0)
    if fam=="linear_svc": return LinearSVC(C=c,loss="squared_hinge",dual="auto",max_iter=20000,tol=1e-4,random_state=0)
    common=dict(C=c,cache_size=512,shrinking=True,probability=False,decision_function_shape="ovr",random_state=0)
    if fam=="svc_linear": return SVC(kernel="linear",**common)
    if fam=="svc_rbf": return SVC(kernel="rbf",gamma=spec["gamma"],**common)
    if fam=="svc_poly": return SVC(kernel="poly",degree=int(spec["degree"]),gamma=spec["gamma"],coef0=float(spec["coef0"]),**common)
    if fam=="svc_sigmoid": return SVC(kernel="sigmoid",gamma=spec["gamma"],coef0=float(spec["coef0"]),**common)
    raise ValueError(fam)

def classical_fit_one(spec,Xtr,ytr,fit_w,Xva,yva,trw,vaw):
    t0=time.time(); caught=[]
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter("always"); m=make_estimator(spec); m.fit(Xtr,ytr,sample_weight=fit_w); trp=m.predict(Xtr); vap=m.predict(Xva); caught=[f"{w.category.__name__}: {w.message}" for w in ws]
    tre=weighted_metrics_from_predictions(ytr,trp,trw); vae=weighted_metrics_from_predictions(yva,vap,vaw)
    return {**dict(spec),"fit_seconds":time.time()-t0,"train_weighted_macro_f1":tre["weighted_macro_f1"],"validation_weighted_macro_f1":vae["weighted_macro_f1"],"validation_weighted_accuracy":vae["weighted_accuracy"],"validation_raw_macro_f1":vae["raw_macro_f1"],"validation_weighted_confusion_matrix":vae["weighted_confusion_matrix"],"warnings":caught}

def run_svd_classical():
    root=OUTPUT_DIR/"svd_classical"
    all_path=root/"all_results.csv"; best_path=root/"best_by_family.csv"; rank_path=root/"best_by_family_and_rank.csv"
    if all(p.exists() for p in [all_path,best_path,rank_path]):
        df=pd.read_csv(all_path); bdf=pd.read_csv(best_path); brdf=pd.read_csv(rank_path)
        expected_all=sum(len(v) for v in SVD_REP_IDS.values())*len(CLASSICAL_MODEL_GRID)
        expected_best=len(TOPICS)*len(CLASSICAL_FAMILIES)
        expected_rank=expected_best*len(SVD_COMPONENTS)
        if len(df)==expected_all and len(bdf)==expected_best and len(brdf)==expected_rank:
            return df,bdf,brdf
    allrows=[]; counter=0; total=sum(len(v) for v in SVD_REP_IDS.values()); t0=time.time()
    for topic in TOPICS:
        tr=TOPIC_DATA[topic]["train"]; va=TOPIC_DATA[topic]["validation"]; ytr=tr.label_id.to_numpy(np.int64); yva=va.label_id.to_numpy(np.int64); fitw=normalized_fit_weights(tr); trw=tr.balance_weight.to_numpy(np.float64); vaw=va.balance_weight.to_numpy(np.float64)
        for rid in SVD_REP_IDS[topic]:
            counter+=1; rp=root/"per_representation"/topic/f"{rid}.csv"; rep=load_svd_rep(topic,rid)
            if rp.exists():
                df=pd.read_csv(rp)
                if len(df)==len(CLASSICAL_MODEL_GRID): allrows.extend(df.to_dict("records")); continue
            Xtr=np.asarray(rep["X_train"],dtype=np.float64); Xva=np.asarray(rep["X_val"],dtype=np.float64); print(f"[SVD classical] {counter}/{total} {topic} {rid}")
            with parallel_config(backend=CLASSICAL_BACKEND,n_jobs=CLASSICAL_N_JOBS,inner_max_num_threads=CLASSICAL_INNER_THREADS):
                recs=Parallel()(delayed(classical_fit_one)(spec,Xtr,ytr,fitw,Xva,yva,trw,vaw) for spec in CLASSICAL_MODEL_GRID)
            for r in recs:r.update({"protocol_hash":PROTOCOL_HASH,"topic":topic,"rep_id":rid,"tfidf_config_id":rep["meta"]["tfidf_config_id"],"svd_components":int(rep["meta"]["svd_components"]),"svd_explained_variance_ratio_sum":float(rep["meta"]["explained_variance_ratio_sum"])})
            rdf=pd.DataFrame(recs); atomic_csv_dump(rdf,rp); allrows.extend(recs); b=rdf.sort_values(["validation_weighted_macro_f1","validation_weighted_accuracy","grid_order"],ascending=[False,False,True]).iloc[0]; print(f"  best={b.family} {b.validation_weighted_macro_f1:.4f}; elapsed={(time.time()-t0)/60:.1f}m")
    df=pd.DataFrame(allrows); atomic_csv_dump(df.sort_values(["topic","rep_id","grid_order"]),root/"all_results.csv")
    best=[]; best_rank=[]
    for topic in TOPICS:
        for fam in CLASSICAL_FAMILIES:
            g=df[(df.topic==topic)&(df.family==fam)]; best.append(g.sort_values(["validation_weighted_macro_f1","validation_weighted_accuracy","rep_id","grid_order"],ascending=[False,False,True,True],kind="mergesort").iloc[0].to_dict())
            for k in SVD_COMPONENTS:
                q=g[g.svd_components==k]; best_rank.append(q.sort_values(["validation_weighted_macro_f1","validation_weighted_accuracy","rep_id","grid_order"],ascending=[False,False,True,True],kind="mergesort").iloc[0].to_dict())
    bdf=pd.DataFrame(best); brdf=pd.DataFrame(best_rank); atomic_csv_dump(bdf,root/"best_by_family.csv"); atomic_csv_dump(brdf,root/"best_by_family_and_rank.csv"); return df,bdf,brdf

SVD_CLASSICAL_ALL,SVD_CLASSICAL_BEST,SVD_CLASSICAL_BY_RANK=run_svd_classical(); display(SVD_CLASSICAL_BEST[["topic","family","rep_id","validation_weighted_macro_f1","validation_weighted_accuracy"]])


,topic,family,rep_id,validation_weighted_macro_f1,validation_weighted_accuracy
0,topicA,logistic_regression,svd__ng12_df10_sub1__k0512,0.905021,0.904839
1,topicA,linear_svc,svd__ng12_df10_sub1__k0256,0.905518,0.905376
2,topicA,svc_linear,svd__ng12_df10_sub1__k0512,0.896645,0.896864
3,topicA,svc_rbf,svd__ng12_df10_sub1__k0064,0.853387,0.853315
4,topicA,svc_poly,svd__ng12_df3_sub1__k0512,0.901684,0.900358
5,topicA,svc_sigmoid,svd__ng12_df5_sub1__k0512,0.900330,0.899910
6,topicB,logistic_regression,svd__ng11_df10_sub1__k0512,0.891752,0.893147
7,topicB,linear_svc,svd__ng11_df10_sub1__k0512,0.890176,0.891290
8,topicB,svc_linear,svd__ng11_df2_sub1__k0512,0.893477,0.893243
9,topicB,svc_rbf,svd__ng11_df10_sub0__k0512,0.777842,0.778753


In [11]:
# 11) Save the selected SVD-classical models and validation outputs

def save_projected_coefficients(model, svd, vec, topic, family, top_n=40):
    if not hasattr(model, "coef_"):
        return
    coef = np.asarray(model.coef_)
    projected = coef @ np.asarray(svd.components_)
    names = np.asarray(vec.get_feature_names_out())
    rows = []
    labels = (
        PORTALS
        if family in {"logistic_regression", "linear_svc"} and projected.shape[0] == 3
        else (
            ["aljazeera_vs_bbc", "aljazeera_vs_guardian", "bbc_vs_guardian"]
            if family == "svc_linear" and projected.shape[0] == 3
            else [f"row_{i}" for i in range(projected.shape[0])]
        )
    )
    for i, label in enumerate(labels):
        order = np.argsort(projected[i])
        for rank, j in enumerate(order[-top_n:][::-1], 1):
            rows.append({
                "topic": topic, "family": family, "coefficient_row": label,
                "direction": "positive", "rank": rank, "feature": names[j],
                "projected_coefficient": float(projected[i, j]),
            })
        for rank, j in enumerate(order[:top_n], 1):
            rows.append({
                "topic": topic, "family": family, "coefficient_row": label,
                "direction": "negative", "rank": rank, "feature": names[j],
                "projected_coefficient": float(projected[i, j]),
            })
    atomic_csv_dump(
        pd.DataFrame(rows),
        OUTPUT_DIR / "svd_classical" / "interpretability" / f"{topic}__{family}__projected_coefficients.csv",
    )


for _, win in SVD_CLASSICAL_BEST.iterrows():
    topic, fam, rid = win.topic, win.family, win.rep_id
    out = OUTPUT_DIR / "svd_classical" / "selected_models" / topic / fam
    bundle_path = out / "model_bundle.joblib"
    metadata_path = out / "metadata.json"
    prediction_path = OUTPUT_DIR / "svd_classical" / "validation_predictions" / f"{topic}__{fam}.csv"
    coefficient_path = OUTPUT_DIR / "svd_classical" / "interpretability" / f"{topic}__{fam}__projected_coefficients.csv"

    # A completed final bundle is reused unchanged on notebook resume.
    if all(p.exists() for p in [bundle_path, metadata_path, prediction_path, coefficient_path]):
        meta = json_load(metadata_path)
        if (
            meta.get("protocol_hash") == PROTOCOL_HASH
            and meta.get("topic") == topic
            and meta.get("family") == fam
            and meta.get("rep_id") == rid
        ):
            print("Reusing selected SVD-classical model:", topic, fam)
            continue

    rep = load_svd_rep(topic, rid)
    tr = TOPIC_DATA[topic]["train"]
    va = TOPIC_DATA[topic]["validation"]
    spec = next(
        s for s in CLASSICAL_MODEL_GRID
        if s["family"] == fam and int(s["grid_order"]) == int(win.grid_order)
    )
    model = make_estimator(spec)
    model.fit(
        np.asarray(rep["X_train"], np.float64),
        tr.label_id.to_numpy(np.int64),
        sample_weight=normalized_fit_weights(tr),
    )
    pred = model.predict(np.asarray(rep["X_val"], np.float64))
    m = weighted_metrics_from_predictions(
        va.label_id.to_numpy(np.int64),
        pred,
        va.balance_weight.to_numpy(np.float64),
    )
    assert abs(m["weighted_macro_f1"] - float(win.validation_weighted_macro_f1)) < 1e-10

    tid = rep["meta"]["tfidf_config_id"]
    td = tfidf_cache_dir(topic, tid)
    sd = svd_cache_dir(topic, rid)
    vec = joblib.load(td / "vectorizer.joblib")
    svd = joblib.load(sd / "svd.joblib")

    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {
            "vectorizer": vec,
            "svd": svd,
            "model": model,
            "metadata": {
                "protocol_hash": PROTOCOL_HASH,
                "topic": topic,
                "family": fam,
                "rep_id": rid,
                "classifier_spec": spec,
            },
        },
        bundle_path,
        compress=3,
    )
    atomic_json_dump(
        {
            "protocol_hash": PROTOCOL_HASH,
            "topic": topic,
            "family": fam,
            "rep_id": rid,
            "classifier_spec": spec,
            "validation_metrics": m,
            "test_evaluated": False,
        },
        metadata_path,
    )
    pdf = va[["article_id", "portal", "year_month", "length_bin", "balance_weight"]].copy()
    pdf["predicted_portal"] = [ID_TO_LABEL[int(x)] for x in pred]
    atomic_csv_dump(pdf, prediction_path)
    save_projected_coefficients(model, svd, vec, topic, fam)


Reusing selected SVD-classical model: topicA logistic_regression
Reusing selected SVD-classical model: topicA linear_svc
Reusing selected SVD-classical model: topicA svc_linear
Reusing selected SVD-classical model: topicB logistic_regression
Reusing selected SVD-classical model: topicB linear_svc
Reusing selected SVD-classical model: topicB svc_linear


## C. Weighted SVD → MLP

The SVD representations are screened in the same staged way, with the final candidate configurations confirmed over five seeds.

In [12]:
# 12) Weighted SVD → MLP final selection
SVD_STAGE1_DF,SVD_STAGE1_SHORT=run_mlp_stage1("svd",SVD_REP_IDS,SVD_STAGE1_KEEP)
SVD_STAGE2_DF,SVD_STAGE2_SHORT=run_mlp_stage2("svd",SVD_STAGE1_SHORT)
SVD_STAGE3_DF=run_mlp_stage3("svd",SVD_STAGE2_SHORT)
SVD_STAGE3_SUMMARY,SVD_SELECTED=aggregate_mlp_stage3("svd",SVD_STAGE3_DF)
display(SVD_STAGE3_SUMMARY); print(json.dumps(SVD_SELECTED,indent=2))


,topic,config_id,rep_id,hidden_layers,dropout,learning_rate,weight_decay,n_parameters,mean_val_weighted_macro_f1,std_val_weighted_macro_f1,mean_val_weighted_accuracy,mean_val_weighted_cross_entropy,mean_best_epoch,config_index
0,topicA,cfg02__svd__ng12_df10_sub1__k0256__h128x64_d0p...,svd__ng12_df10_sub1__k0256,"[128, 64]",0.0,0.0010,0.000,41347,0.918615,0.002381,0.918011,0.266975,21.2,1
1,topicA,cfg01__svd__ng12_df10_sub1__k0256__h128x64_d0p...,svd__ng12_df10_sub1__k0256,"[128, 64]",0.0,0.0010,0.001,41347,0.918462,0.002494,0.917742,0.269310,20.6,0
2,topicA,cfg04__svd__ng12_df10_sub1__k0256__h128x64_d0p...,svd__ng12_df10_sub1__k0256,"[128, 64]",0.3,0.0010,0.000,41347,0.917636,0.002282,0.916953,0.259858,27.6,3
3,topicA,cfg03__svd__ng12_df10_sub1__k0256__h128x64_d0p...,svd__ng12_df10_sub1__k0256,"[128, 64]",0.3,0.0010,0.001,41347,0.917636,0.002282,0.916953,0.259944,27.6,2
4,topicA,cfg06__svd__ng12_df10_sub1__k0512__h128x64_d0p...,svd__ng12_df10_sub1__k0512,"[128, 64]",0.3,0.0003,0.001,74115,0.911348,0.006478,0.910842,0.304332,38.6,5
5,topicA,cfg05__svd__ng12_df10_sub1__k0512__h128x64_d0p...,svd__ng12_df10_sub1__k0512,"[128, 64]",0.3,0.0003,0.000,74115,0.911348,0.006478,0.910842,0.304355,38.6,4
6,topicB,cfg03__svd__ng11_df10_sub1__k0512__h128x64_d0p...,svd__ng11_df10_sub1__k0512,"[128, 64]",0.3,0.0003,0.000,74115,0.894674,0.004322,0.895152,0.479683,41.4,2
7,topicB,cfg04__svd__ng11_df10_sub1__k0512__h128x64_d0p...,svd__ng11_df10_sub1__k0512,"[128, 64]",0.3,0.0003,0.001,74115,0.894440,0.004620,0.894972,0.479912,43.2,3
8,topicB,cfg01__svd__ng11_df5_sub1__k0512__h128_d0p3_lr...,svd__ng11_df5_sub1__k0512,[128],0.3,0.0010,0.000,66051,0.888010,0.004405,0.888565,0.478434,31.4,0
9,topicB,cfg02__svd__ng11_df5_sub1__k0512__h128_d0p3_lr...,svd__ng11_df5_sub1__k0512,[128],0.3,0.0010,0.001,66051,0.888010,0.004405,0.888565,0.478490,31.4,1


{
  "topicA": {
    "topic": "topicA",
    "config_id": "cfg02__svd__ng12_df10_sub1__k0256__h128x64_d0p0_lr1em03_wd0e00",
    "rep_id": "svd__ng12_df10_sub1__k0256",
    "hidden_layers": [
      128,
      64
    ],
    "dropout": 0.0,
    "learning_rate": 0.001,
    "weight_decay": 0.0,
    "n_parameters": 41347,
    "mean_val_weighted_macro_f1": 0.9186151309162813,
    "std_val_weighted_macro_f1": 0.0023810181969440088,
    "mean_val_weighted_accuracy": 0.918010752688172,
    "mean_val_weighted_cross_entropy": 0.26697498329863484,
    "mean_best_epoch": 21.2,
    "config_index": 1
  },
  "topicB": {
    "topic": "topicB",
    "config_id": "cfg03__svd__ng11_df10_sub1__k0512__h128x64_d0p3_lr3em04_wd0e00",
    "rep_id": "svd__ng11_df10_sub1__k0512",
    "hidden_layers": [
      128,
      64
    ],
    "dropout": 0.3,
    "learning_rate": 0.0003,
    "weight_decay": 0.0,
    "n_parameters": 74115,
    "mean_val_weighted_macro_f1": 0.8946739797461054,
    "std_val_weighted_macro_f1": 0.0

In [13]:
# 13) Finalize selected direct and SVD MLP bundles/checkpoints

def finalize_mlp_family(family, selected, stage3_df):
    result = {}
    for topic in TOPICS:
        w = selected[topic]
        rid = w["rep_id"]
        out = OUTPUT_DIR / "selected_mlp_models" / family / topic
        out.mkdir(parents=True, exist_ok=True)

        metadata_path = out / "metadata.json"
        bundle_path = out / "representation_bundle.joblib"
        expected_checkpoints = [out / f"mlp_seed_{int(seed)}.pt" for seed in CONFIRMATION_MASTER_SEEDS]

        # A completed selected model is reused unchanged on notebook resume.
        if metadata_path.exists() and bundle_path.exists() and all(p.exists() for p in expected_checkpoints):
            meta = json_load(metadata_path)
            selected_cfg = meta.get("selected_config", {})
            if (
                meta.get("protocol_hash") == PROTOCOL_HASH
                and meta.get("family") == family
                and meta.get("topic") == topic
                and selected_cfg.get("config_id") == w.get("config_id")
                and selected_cfg.get("rep_id") == w.get("rep_id")
            ):
                result[topic] = meta
                print("Reusing selected MLP model:", family, topic)
                continue

        rep = load_representation(topic, rid)
        tid = rep["meta"]["tfidf_config_id"]
        vec = joblib.load(tfidf_cache_dir(topic, tid) / "vectorizer.joblib")
        if family == "direct":
            bundle = {"vectorizer": vec, "representation_family": "direct"}
        else:
            bundle = {
                "vectorizer": vec,
                "svd": joblib.load(svd_cache_dir(topic, rid) / "svd.joblib"),
                "representation_family": "svd",
            }
        joblib.dump(bundle, bundle_path, compress=3)

        runs = stage3_df[
            (stage3_df.topic == topic) & (stage3_df.config_id == w["config_id"])
        ].sort_values("master_seed")
        assert set(runs.master_seed.astype(int)) == set(CONFIRMATION_MASTER_SEEDS)

        copied = []
        for _, rr in runs.iterrows():
            src = OUTPUT_DIR / rr.checkpoint_path
            dst = out / f"mlp_seed_{int(rr.master_seed)}.pt"
            shutil.copy2(src, dst)
            copied.append(str(dst.relative_to(OUTPUT_DIR)))

        meta = {
            "protocol_hash": PROTOCOL_HASH,
            "family": family,
            "topic": topic,
            "selected_config": w,
            "representation_metadata": rep["meta"],
            "selected_checkpoints": copied,
            "confirmation_master_seeds": CONFIRMATION_MASTER_SEEDS,
            "test_evaluated": False,
        }
        atomic_json_dump(meta, metadata_path)
        result[topic] = meta
    return result


DIRECT_FINAL = finalize_mlp_family("direct", DIRECT_SELECTED, DIRECT_STAGE3_DF)
SVD_FINAL = finalize_mlp_family("svd", SVD_SELECTED, SVD_STAGE3_DF)


Reusing selected MLP model: direct topicA
Reusing selected MLP model: direct topicB
Reusing selected MLP model: svd topicA
Reusing selected MLP model: svd topicB


In [14]:
# 14) Five-seed training-matched PyTorch linear-head ablation for BOTH selected representations

def run_linear_ablation_family(family,selected):
    stage=f"diagnostics_torch_linear_{family}"
    for topic in TOPICS:
        w=selected[topic]; rep=load_representation(topic,w["rep_id"])
        for si,seed in enumerate(CONFIRMATION_MASTER_SEEDS):
            run_id=f"{topic}__{w['rep_id']}__linear__seed{seed}"; rp=run_record_path(stage,run_id)
            if rp.exists():continue
            rec=train_one_run(topic,rep,[],0.0,float(w["learning_rate"]),float(w["weight_decay"]),seed,run_id,history_path(stage,run_id)); rec.update({"stage":stage,"grid_order":si,"seed_index":si}); atomic_json_dump(rec,rp)
    df=load_stage_records(stage); atomic_csv_dump(df.sort_values(["topic","seed_index"]),OUTPUT_DIR/stage/"seed_runs.csv"); summary=df.groupby("topic").agg(mean_val_weighted_macro_f1=("best_val_weighted_macro_f1","mean"),std_val_weighted_macro_f1=("best_val_weighted_macro_f1","std"),mean_val_weighted_accuracy=("best_val_weighted_accuracy","mean")).reset_index(); summary["family"]=family; atomic_csv_dump(summary,OUTPUT_DIR/stage/"summary.csv"); return df,summary

if RUN_TORCH_LINEAR_ABLATION:
    DIRECT_LINEAR_RUNS,DIRECT_LINEAR_SUMMARY=run_linear_ablation_family("direct",DIRECT_SELECTED); SVD_LINEAR_RUNS,SVD_LINEAR_SUMMARY=run_linear_ablation_family("svd",SVD_SELECTED); TORCH_LINEAR_SUMMARY=pd.concat([DIRECT_LINEAR_SUMMARY,SVD_LINEAR_SUMMARY],ignore_index=True); display(TORCH_LINEAR_SUMMARY)
else: TORCH_LINEAR_SUMMARY=pd.DataFrame()


,topic,mean_val_weighted_macro_f1,std_val_weighted_macro_f1,mean_val_weighted_accuracy,family
0,topicA,0.877339,0.005349,0.876523,direct
1,topicB,0.748361,0.017012,0.748386,direct
2,topicA,0.861259,0.010529,0.860950,svd
3,topicB,0.843689,0.020904,0.843371,svd


## Final summaries

The final cells save the selected configurations, compact comparison tables and plots used by the later test-evaluation notebook.

In [15]:
# 15) Consolidated comparison plots and selection index
# Direct-vs-SVD MLP comparison.
rows=[]
for topic in TOPICS:
    rows += [{"topic":topic,"family":"Direct TF-IDF MLP","score":DIRECT_SELECTED[topic]["mean_val_weighted_macro_f1"]},{"topic":topic,"family":"SVD MLP","score":SVD_SELECTED[topic]["mean_val_weighted_macro_f1"]}]
MLP_COMPARISON=pd.DataFrame(rows); atomic_csv_dump(MLP_COMPARISON,OUTPUT_DIR/"diagnostics"/"direct_vs_svd_mlp.csv")
for topic in TOPICS:
    s=MLP_COMPARISON[MLP_COMPARISON.topic==topic]; fig,ax=plt.subplots(figsize=(6.5,4.5)); ax.bar(s.family,s.score); ax.set_ylabel("Validation weighted macro-F1"); ax.set_title(f"{topic}: direct vs SVD MLP"); save_figure(fig,OUTPUT_DIR/"plots"/f"{topic}_direct_vs_svd_mlp.png"); plt.close(fig)
# Classical best-by-rank curves.
for topic in TOPICS:
    fig,ax=plt.subplots(figsize=(8,5))
    for fam in CLASSICAL_FAMILIES:
        s=SVD_CLASSICAL_BY_RANK[(SVD_CLASSICAL_BY_RANK.topic==topic)&(SVD_CLASSICAL_BY_RANK.family==fam)].sort_values("svd_components"); ax.plot(s.svd_components,s.validation_weighted_macro_f1,marker="o",label=fam)
    ax.set_xlabel("SVD components"); ax.set_ylabel("Best validation weighted macro-F1"); ax.set_title(f"{topic}: SVD rank and classical performance"); ax.legend(fontsize=8); save_figure(fig,OUTPUT_DIR/"plots"/f"{topic}_svd_rank_classical.png"); plt.close(fig)


# Compact selection index used by the final evaluation notebook.
_selection_index_path = OUTPUT_DIR/"FINAL_SELECTION_INDEX.json"
if not _selection_index_path.exists():
    atomic_json_dump({"protocol_hash":PROTOCOL_HASH,"direct_mlp":DIRECT_SELECTED,"svd_mlp":SVD_SELECTED,"svd_classical_best":SVD_CLASSICAL_BEST.to_dict("records"),"test_evaluated":False}, _selection_index_path)
print("Model-selection outputs saved to", OUTPUT_DIR)


Model-selection outputs saved to /home/dorjan/NLP/nlp_portal_project/modeling_results/tfidf_neural_svd_final


In [16]:
# 16) Compact final notebook summary
summary=[]
for topic in TOPICS:
    summary.append({"topic":topic,"family":"direct_mlp","rep_id":DIRECT_SELECTED[topic]["rep_id"],"val_wMacroF1":DIRECT_SELECTED[topic]["mean_val_weighted_macro_f1"],"sd":DIRECT_SELECTED[topic]["std_val_weighted_macro_f1"]})
    summary.append({"topic":topic,"family":"svd_mlp","rep_id":SVD_SELECTED[topic]["rep_id"],"val_wMacroF1":SVD_SELECTED[topic]["mean_val_weighted_macro_f1"],"sd":SVD_SELECTED[topic]["std_val_weighted_macro_f1"]})
display(pd.DataFrame(summary)); display(SVD_CLASSICAL_BEST[["topic","family","rep_id","validation_weighted_macro_f1"]].sort_values(["topic","family"]))
print("Model selection complete; the test set was not used in this notebook.")


,topic,family,rep_id,val_wMacroF1,sd
0,topicA,direct_mlp,direct__ng12_df10_sub1,0.902144,0.004218
1,topicA,svd_mlp,svd__ng12_df10_sub1__k0256,0.918615,0.002381
2,topicB,direct_mlp,direct__ng12_df10_sub0,0.812422,0.004639
3,topicB,svd_mlp,svd__ng11_df10_sub1__k0512,0.894674,0.004322


,topic,family,rep_id,validation_weighted_macro_f1
1,topicA,linear_svc,svd__ng12_df10_sub1__k0256,0.905518
0,topicA,logistic_regression,svd__ng12_df10_sub1__k0512,0.905021
2,topicA,svc_linear,svd__ng12_df10_sub1__k0512,0.896645
4,topicA,svc_poly,svd__ng12_df3_sub1__k0512,0.901684
3,topicA,svc_rbf,svd__ng12_df10_sub1__k0064,0.853387
5,topicA,svc_sigmoid,svd__ng12_df5_sub1__k0512,0.900330
7,topicB,linear_svc,svd__ng11_df10_sub1__k0512,0.890176
6,topicB,logistic_regression,svd__ng11_df10_sub1__k0512,0.891752
8,topicB,svc_linear,svd__ng11_df2_sub1__k0512,0.893477
10,topicB,svc_poly,svd__ng11_df10_sub1__k0512,0.885844


Model selection complete; the test set was not used in this notebook.
